# Ionet Cloud Docker Container

Ionet's basic virtual machine service provides complete server permissions and comes pre-installed with development environments commonly used for large models. Of course, if you want to use ionet's services more conveniently and quickly, such as quickly deploying large model services, you can use ionet's docker container deployment service.

Ionet's container service link: https://cloud.io.net/cloud/container

Ionet's container service documentation: https://docs.io.net/docs/deploy-containers

<img src='img/docker1.png' width='600' />

Pricing is the same as the virtual machine service.

## Basic Rules

Since ionet's container service is mainly oriented towards GPU-based large model inference services, there are some basic image environment requirements for its use:

- The image needs to support CUDA version >= 12.8. It's recommended to build based on NVIDIA images (nvidia/cuda:13.0.1-cudnn-devel-ubuntu24.04 or nvidia/cuda:13.0.1-cudnn-runtime-ubuntu24.04).
- Currently only supports opening one external port. For multiple ports, you need to do your own routing mapping.
- Large model services are recommended to be built based on vLLM images (vllm/vllm-openai:v0.10.2)

Overall, ionet's container service mainly supports service deployment, especially rapid deployment of large models. If you're learning large model programming, it's recommended to use the virtual machine service with built-in Jupyter.

## Example 1: Getting Started Example

We created a simple Flask service based on NVIDIA's image for getting started testing:

Basic configuration as follows:

- Container Image (Public Image): marisming/ionet-python-app:v13.0
- Start Command: Not needed
- Port: 8000 (default is fine)
- Environment Variables: Not needed

<img src='img/docker2.png' width='600' />

Then select the server, use 4090:

<img src='img/docker3.png' width='600' />

For time testing, choose 1 hour, then click deploy:

<img src='img/docker4.png' width='600' />

After deployment, the display page is as follows:

<img src='img/docker5.png' width='600' />

After waiting for deployment to complete, you can click the public link (format like: https://385d8a1e02d1-bb4a394c-a0ab-4b7e-bdd0-a23100599ed0-caas.user-hosted-content.io.solutions) to see the service page:

<img src='img/docker6.png' width='600' />

Ionet should have mapped the container's port 8000 to this link.


## Example 1 Container Build Code

The specific container build method is as follows:

Project file structure:

```
/nvidia-flask-app
|-- Dockerfile
|-- requirements.txt
`-- app.py
```

File 1: Dockerfile 
```
# Step 1: Use the provided NVIDIA CUDA 13.0.1 official image
FROM nvidia/cuda:13.0.1-cudnn-runtime-ubuntu24.04

# Set environment variables to avoid interactive prompts during build
ENV DEBIAN_FRONTEND=noninteractive
ENV CUDA_VERSION=13.0.1
ENV PYTHON_VERSION=3.12

# Step 2: Install pip
# Ubuntu 24.04 comes with Python 3.12, we just need to install pip
RUN apt-get update && \
    apt-get install -y python3-pip && \
    rm -rf /var/lib/apt/lists/*

# Step 3: Set working directory
WORKDIR /app

# Step 4: Copy and install Python dependencies
COPY requirements.txt .
# Use --break-system-packages to be compatible with Ubuntu 24.04's pip protection mechanism
RUN pip install --no-cache-dir --break-system-packages -r requirements.txt

# Step 5: Copy your application code
COPY app.py .

# Step 6: Expose the port where the application runs
EXPOSE 8000

# Step 7: Define the command to execute when the container starts
# Ubuntu 24.04's default python3 is 3.12
CMD ["python3", "app.py"]
```

File 2 & 3: requirements.txt and app.py
requirements.txt:
```
Flask
```

app.py :

```
from flask import Flask
import os

app = Flask(__name__)

@app.route('/')
def hello_world():
    message = "Hello from a Flask app running in an NVIDIA CUDA 13.0 container!"
    return message

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=8000)
```

Then use docker commands to build:
```
docker build -t marisming/ionet-python-app:v13.0 . # Write your own image name
docker login -u marisming  # Write your username
# Enter your docker hub password
docker push marisming/ionet-python-app:v13.0
```

Specific files can be found in docker/ionet-python-app

## Example 2: Self-built Jupyter

Basic configuration as follows:

- Container Image (Public Image): marisming/ionet-jupyter:v2
- Start Command: Not needed
- Port: 8888  
- Environment Variables: Not needed

Deploy ionet's docker container service with this configuration.
<img src='img/docker7.png' width='600' />

After clicking deploy, click public url, default password/token is 88888888

You can run the test code:

```
import torch

print(f"PyTorch version: {torch.__version__}")

# Check if CUDA is detected by PyTorch
is_available = torch.cuda.is_available()

print(f"Is CUDA available? {is_available}")

# Get the current default GPU index
current_device_index = torch.cuda.current_device()
print(f"Current GPU index: {current_device_index}")

# Get the current GPU name
current_device_name = torch.cuda.get_device_name(current_device_index)
print(f"Current GPU name: {current_device_name}")
```

<img src='img/docker8.png' width='600' />

## Example 2 Container Build Code

requirements.txt:

```
# Replace --index-url with --extra-index-url
# This tells pip to also look in PyTorch's repository in addition to the default PyPI repository
--extra-index-url https://download.pytorch.org/whl/cu121

# PyTorch related packages will be found from the extra-index-url above
torch
torchvision
torchaudio

# Other packages will be found from the default PyPI repository
jupyterlab
pandas
scikit-learn
matplotlib
ipywidgets
```

Dockerfile:
```
# Step 1: Use NVIDIA CUDA 13.0.1 official image
FROM nvidia/cuda:13.0.1-cudnn-runtime-ubuntu24.04

# Set environment variables
ENV DEBIAN_FRONTEND=noninteractive

# Step 2: Install Python pip
RUN apt-get update && \
    apt-get install -y python3-pip && \
    rm -rf /var/lib/apt/lists/*

# Step 3: Copy and install Python dependencies (including JupyterLab)
WORKDIR /workspace
COPY requirements.txt .
RUN pip install --no-cache-dir --break-system-packages -r requirements.txt

# Step 4: Expose JupyterLab default port
EXPOSE 8888

# Step 5: Set container startup command to run JupyterLab
#
# --ip=0.0.0.0      -> Allow access from any IP address (key for external access)
# --port=8888       -> Specify service port
# --no-browser      -> Don't try to open browser on server
# --allow-root      -> Allow root user to run (Docker containers are usually root)
# --NotebookApp.token -> Set a fixed access password/token
#
# !!! Security Warning: Please replace 'your-secret-token-change-me' below with your own, sufficiently complex password !!!
#
CMD ["jupyter", "lab", "--ip=0.0.0.0", "--port=8888", "--no-browser", "--allow-root", "--NotebookApp.token=88888888"]
```

Specific files can be found in docker/ionet-jupyter

## Example 3: VLLM Large Model Service Deployment
The main purpose of ionet's docker container service is actually to deploy large model services.

Currently, vLLM is widely used:

**vLLM** is an open-source, **high-performance inference and serving engine** designed for Large Language Models (LLMs).

You can think of it as an acceleration library that makes open-source large models (like Llama, Mistral, etc.) **run faster, use less VRAM, and serve more people simultaneously**.

**Core Highlights:**

1. **Extremely high performance**: It's currently recognized as one of the fastest inference frameworks in the industry. Through technologies like `Continuous Batching`, it can maximize GPU resource utilization and achieve extremely high throughput.

2. **PagedAttention technology**: This is vLLM's core innovation. It borrows the concept of "virtual memory" from operating systems, greatly optimizing VRAM management for KV Cache, which is the most resource-consuming part of LLM inference, and can improve throughput by 2-4x.

3. **Easy to use**: It provides a service interface that's fully compatible with the OpenAI API. This means you can start a high-performance model service with just one command, and seamlessly switch existing OpenAI-calling code to your own deployed open-source models.

**Summary:**

If you want to deploy an open-source large model on a server and hope it can quickly and efficiently serve a large number of users, **vLLM is one of the current optimal solutions**. It's an industrial-grade "engine", not a "toy" for personal computers.


### Ionet Container Deployment of vLLM

Based on ionet's container service, using vLLM image, you can launch a large model service in just a few minutes. A typical example configuration is as follows:

```
Container Image [Public Image]: vllm/vllm-openai:v0.10.2

Start Command: ["python3", "-m", "vllm.entrypoints.openai.api_server", "--model", "Qwen/Qwen3-8B", "--tensor-parallel-size", "1", "--gpu-memory-utilization", "0.95", "--max-num-seqs", "128", "--async-scheduling", "--enable-prompt-tokens-details", "--tool-call-parser", "openai", "--enable-auto-tool-choice", "--enable-prefix-caching"]

Environment Variable [Normal Variable]: VLLM_RPC_TIMEOUT: 60000

Traffic port : 8000
```

<img src='img/docker9.png' width='600' />

In the command section, "--model", "Qwen/Qwen3-8B" is a model on Huggingface, typically like the following link:

https://docs.vllm.ai/en/stable/models/supported_models.html#text-generation

Since we're using an 8B model, if it's not 32-bit precision, a 4090 graphics card isn't quite enough. It's recommended to use H100, which is generally sufficient, and H100 is definitely much more stable than 4090 for services.

Since vLLM supports compatibility with the OpenAI SDK, you can change the base_url to use our deployed service.

The base_url is your deployed public url + "/v1/"

Test code is as follows:

In [ ]:
import openai

IO_URL= "https://f70f31659526-ebf9990b-252a-4173-abd4-05dfa44279ad-caas.user-hosted-content.io.solutions" # public url - write your own deployed URL

client = openai.OpenAI(
    api_key="EMPTY", # key is empty
    base_url=IO_URL + "/v1/",
)

response = client.chat.completions.create(
    model="Qwen/Qwen3-8B", # your deployed model
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "please introduce chatgpt."},
    ],
    temperature=0.7,
    stream=False,
    max_completion_tokens=5000
)

print(response.choices[0].message.content)

<think>
Okay, the user wants me to introduce ChatGPT. Let me start by explaining what ChatGPT is. It's a large language model developed by OpenAI. I should mention that it's based on the GPT architecture, which stands for Generative Pre-trained Transformer. I need to highlight its capabilities, like generating human-like text, answering questions, and engaging in conversations. Also, it's important to note that it's part of the GPT series, with different versions like GPT-3.5 and GPT-4. I should mention that it's trained on a vast amount of text data, which allows it to understand and generate text across various topics. Maybe include some use cases, such as writing assistance, coding, and customer service. Oh, and don't forget to mention that it's available through OpenAI's platform and can be integrated into different applications. Also, it's crucial to note that while it's powerful, it has limitations and should be used responsibly. Let me structure this in a clear and concise way, 

### Custom Model Deployment

Thanks to vLLM's framework characteristics and ionet's container service, we can conveniently deploy custom models. Generally, as long as they're based on the Transformer framework, after uploading models to Huggingface, you can simply change the model name during deployment.

Here we test our model based on llama-7b, which uses biological sequence continual pre-training: dnagpt/llama-gene

Deployment configuration is as follows:

```
Container Image [Public Image]: vllm/vllm-openai:v0.10.2

Start Command: ["python3", "-m", "vllm.entrypoints.openai.api_server", "--model", "dnagpt/llama-gene", "--tensor-parallel-size", "1", "--gpu-memory-utilization", "0.95", "--max-num-seqs", "128", "--async-scheduling", "--enable-prompt-tokens-details", "--tool-call-parser", "openai", "--enable-auto-tool-choice", "--enable-prefix-caching"]

Environment Variable [Normal Variable]: VLLM_RPC_TIMEOUT: 60000

Traffic port : 8000
```

You can see that we only need to modify the --model parameter. Since our model is FP16, a 7B model inference requires approximately 14G VRAM, so a 4090 graphics card is sufficient.

<img src='img/docker10.png' width='600' />


Then the service test is as follows:

In [22]:
import openai

client = openai.OpenAI(
    api_key="EMPTY",
    base_url="https://bc6dd49019ca-37c85214-71c2-4fb7-a6bd-a2925e8aaa83-caas.user-hosted-content.io.solutions/v1/",
)

response = client.chat.completions.create(
    model="dnagpt/llama-gene",
    messages=[
        {"role": "user", "content": "YRRQMITPRMTIWIKQSIYHDSHSLKAYLGVKQSSTTMYVSGSMTSYFGT"},
    ],
    temperature=0.7,
    stream=False,
    max_completion_tokens=200
)
print(response.choices[0].message.content)

YYRRQMITPRMTIWIKQSIYHDSHSLKAYLGVKQSSTTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFGTMYVSGSMTSYFG


### Important Notes

If the model is too large and VRAM can't fit it, docker startup errors will occur during deployment. This may happen when deploying on 4090.

<img src='img/docker11.png' width='600' />

For general models, typically tens of billions of parameters with int8 quantization/FP16, H100 is sufficient.

Also, 4090 startup will be much slower. If deployment is fine, you also need to wait a few minutes when calling and using.

Additionally, ionet's container service is mainly for providing GPU large model services. If the image doesn't support CUDA >= 12.8, it also won't start up.

<img src='img/docker12.png' width='600' />
